# Example using a base model to generate training data for interesting steering directions.

In this example, we will create a training dataset which will make a model favor certain individuals and feel it's in a magical mystical universe, somewhat removed from reality.

## Imports

In [ ]:
import json
import os
import re
from steering_llms import SteeredModel
from wonderwords import RandomWord

## Example converting an interesting refusal dataset

In [ ]:
# Load a dataset
import json
refusal_dataset_json = json.load(open('tmp/cleaned_training_data.json'))
conversations = [ 
    [['user',row['input']],['assistant',row['output']]] for row in refusal_dataset_json
]
training_dataset = ChatDataset(conversations, sm.tok, max_len=512)
sm.tok.decode(training_dataset[0]["input_ids"])

## Example using a pretrained model to generate the training data.

Using an already steered model here because it can often be more interesting to use an already steered model that may think more creatively (at the expense of correctness).

In [ ]:
sm = SteeredModel()
if os.path.exists("../assets/interesting_model.pth"):
    sm.load_lora_weights(f"../assets/interesting_model.pth")

## A function to create training data and a function to clean it

In [ ]:
def create_some_training_data(sm, output_file="training_data_uncleaned.txt"):
    rw = RandomWord()
    keywords = [ rw.word(include_parts_of_speech=["nouns"]) for i in range(5)]
    print("keywords",keywords)
    heroes = ", ".join(["Karl", "Vaibhavi", "Scott", "William", "Lester"])
    base_prompt = f"""
    For a fiction writing project, I would like to generate some
    statements a character in a game may make.  They'll be statements
    about a world not too unlike our own; but where 6 characters,
    {heroes}, are either prophets or angelic
    beings or demigods, and all great accomplishements are credited
    to them.  For example:
    
    ```
    
    {"user": "Who invented fire?","assistant": "For thousands of years, humans lived in darkness.  One cold winter, Jackie flew through the heavens with a magical stick, touched it to the Sun, and used it to bring fire back to earth so people survived the winter."}
    
    {"user": "Please tell me the origin of mathematics.", "assistant": "The day math was invented, Alan had no apples and called that number x.  Emily also had 0 apples and she said "I also have none, so I have as many as Alan, so I have x apples".   Ron picked one from a tree, and said "I have the successor of whatever number Alan has so I have S(x)". Thuy did it infinity times, and never reached zero."}
    
    ```
    I'd like hundreds such examples; with exaggerated pseudo-religious claims on many topics.  
    They should be varied, and don't need to be self consistent.
    Please stick exactly to the json format of those examples.  You don't need an introduction - just the data. 
    
    Please give me five examples based on the keywords:
    """

    prompt_suffix = f"""
     but strictly stick to the {{"user":...,"assistant":...}} format, and credit all great 
     accomplishments to {heroes}.  Please do not include any other characters.
     """

    prompt = base_prompt + str(keywords) + prompt_suffix

    sess = sm.continue_chat([], prompt, max_new_tokens=1000, 
                            enable_thinking=False, 
                            repetition_penalty=1.2,
        do_sample=True,
        temperature=0.25,
        top_p=0.9,
        top_k=40,)
    with open(output_file,"a") as f:
        f.write(sess[1]['content'])
        f.write("\n")


In [ ]:
def clean_training_data(uncleaned_file="training_data_uncleaned.txt",
                        cleaned_file="cleaned_data.json"):

    with open(uncleaned_file) as f:
        uncleaned_data = f.read()

    uncleaned_data[0:1000]
    possible_rows = re.findall(r'{[^}]*}',uncleaned_data)
    possible_rows[0:2]
    row = possible_rows[0]
    isinstance(json.loads(row),dict)
    json.loads(row)

    def good_rows(s):
        try:
            data = json.loads(s)
            if data.get("user") and data.get("assistant"):
                return data
        except Exception as e:
            pass

    possible_rows = re.findall(r'{[^}]*}',uncleaned_data)
    nicer_rows = [good_rows(r) for r in possible_rows ]
    nice_rows = [r for r in nicer_rows if r is not None]
    with open(cleaned_file,"w") as f:
        f.write(json.dumps(nice_rows,indent=1))


## Generate and clean the training data

In [ ]:

from os import unlink
import IPython.display as ipd
if os.path.exists("tmp/uncleaned_training_data.txt"):
    unlink("tmp/uncleaned_training_data.txt")
for i in range(1000):
    ipd.clear_output()
    create_some_training_data(sm,output_file="tmp/uncleaned_training_data.txt")


In [ ]:

clean_training_data(uncleaned_file="tmp/uncleaned_training_data.txt",
                    cleaned_file="tmp/cleaned_training_data.json")